# 02. Feature Engineering
EDA 에서 확인한 피처 정의(`src/features.py`)로 셀당 한 행의 피처 표를 만든다.
- ΔQ(V) = Qdlin[사이클 100] − Qdlin[사이클 10] 의 통계 (같은 셀 안의 차이 → 배치 간 Qdlin 시작점 차이가 상쇄)
- 초기 100 사이클 요약값, 충전 프로토콜 분해
- 셀 하나 안에서만 계산하므로 다른 셀·배치 정보를 쓰지 않는다 (스케일링·결측 대치는 학습 데이터로만 fit — `src/train.py`)

In [1]:
import sys, platform, random
from pathlib import Path
ROOT = Path.cwd().parent            # notebooks/ 안에서 실행한다고 가정 (프로젝트 루트 = 한 단계 위)
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, scipy, sklearn
from src.train import RANDOM_STATE
random.seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)      # 무작위성이 있는 모든 곳에 같은 seed

# 환경·버전 기록 (재현성)
print('python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__,
      '| scipy', scipy.__version__, '| scikit-learn', sklearn.__version__, '| RANDOM_STATE', RANDOM_STATE)

python 3.11.15 | numpy 2.4.6 | pandas 3.0.6 | scipy 1.17.1 | scikit-learn 1.9.1 | RANDOM_STATE 42


In [2]:
from src.preprocess import load_cells, build_cohort
from src.features import build_feature_table

cohort = build_cohort(load_cells())              # 처음 한 번은 .mat 을 읽어 캐시를 만든다 (배치당 20~30초)
feat = build_feature_table(cohort)
print(feat.shape, feat.groupby('batch').size().to_dict(), '| censored(수명 미도달):', int(feat.censored.sum()))
feat.head()

(131, 22) {'b1': 46, 'b2': 39, 'b3': 46} | censored(수명 미도달): 2


,cid,batch,policy,newstructure,cycle_life,censored,end_QD,label_uncertain,dq_logvar,dq_logmean,...,dq_kurt,QD_slope,IR,Tavg,Tmax,chargetime,C1,switch,C2,avgC
0,b1c0,b1,3.6C(80%)-3.6C,False,1190.0,False,1.026688,True,-5.051527,-2.553712,...,-0.909997,0.016336,0.016619,31.566972,35.353580,13.358212,3.6,80.0,3.6,3.6
1,b1c1,b1,3.6C(80%)-3.6C,False,1179.0,False,1.038721,True,-5.135776,-2.428027,...,-1.038547,0.005970,0.016923,31.258835,34.137924,13.341623,3.6,80.0,3.6,3.6
2,b1c2,b1,3.6C(80%)-3.6C,False,1177.0,False,1.043592,True,-4.951883,-2.383163,...,-1.173739,0.010296,0.016769,31.406702,34.527420,13.341570,3.6,80.0,3.6,3.6
3,b1c3,b1,4C(80%)-4C,False,1226.0,False,0.972497,True,-4.386348,-2.104748,...,-1.124219,0.014360,0.016315,29.847548,30.585636,12.092243,4.0,80.0,4.0,4.0
4,b1c4,b1,4C(80%)-4C,False,1227.0,False,1.021984,True,-4.604779,-2.202790,...,-1.361164,0.016663,0.016677,31.367527,34.341938,12.092205,4.0,80.0,4.0,4.0


## 점검 (타깃 누수 / 결측)
- 피처는 모두 초기 100 사이클에서만 계산했고 `cycle_life` 에서 파생된 변수는 없다.
- 결측은 `IR` 에 있다 (Batch 2 일부 셀은 초기 IR 이 전부 0 → 결측 처리). 대치는 파이프라인에서 학습 데이터 중앙값으로 한다.

In [3]:
feature_cols = [c for c in feat.columns if c not in ['cid','batch','policy','newstructure','cycle_life','censored']]
print('피처:', feature_cols)
print('결측:', feat[feature_cols].isna().sum()[lambda s: s > 0].to_dict())
print('타깃(cycle_life)과 이름이 겹치는 피처:', [c for c in feature_cols if 'life' in c])

피처: ['end_QD', 'label_uncertain', 'dq_logvar', 'dq_logmean', 'dq_logmin', 'dq_skew', 'dq_kurt', 'QD_slope', 'IR', 'Tavg', 'Tmax', 'chargetime', 'C1', 'switch', 'C2', 'avgC']
결측: {'IR': 6}
타깃(cycle_life)과 이름이 겹치는 피처: []


In [4]:
(ROOT / 'results').mkdir(exist_ok=True)
feat.to_csv(ROOT / 'results' / 'features.csv', index=False)      # 피처 표 스냅샷 (재현성)
print('saved results/features.csv', feat.shape)

saved results/features.csv (131, 22)
